In [25]:
import json
import numpy as np
import torch
import torch.nn as nn

In [4]:
def load_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

In [5]:
vocab_data = load_json("vocab.json")

In [6]:
index_to_char = {index: char for index, char in enumerate(vocab_data)}
char_to_index = {char: index for index, char in enumerate(vocab_data)}

In [37]:
embedding = np.array(load_json("embedding.weight.json"), dtype=np.float32)
W_h = np.array(load_json("W_h.weight.json"), dtype=np.float32)
W_h_bias = np.array(load_json("W_h.bias.json"), dtype=np.float32).squeeze()
U_h = np.array(load_json("U_h.weight.json"), dtype=np.float32)
W_y = np.array(load_json("W_y.weight.json"), dtype=np.float32)

In [38]:
if embedding.shape == (48, 133):
    embedding = embedding.T

if W_h.shape == (48, 160):
    W_h = W_h.T

if W_y.shape == (160, 133):
    W_y = W_y.T

In [40]:
start_token = "["
end_token = "]"

In [41]:
current_index = char_to_index[start_token]
h_t = np.zeros(160, dtype=np.float32)

In [42]:
decoded_chars = []
max_steps = 500

In [44]:
for step in range(max_steps):
    x_t = embedding[current_index]
    h_t = np.maximum(0, W_h @ x_t + W_h_bias + U_h @ h_t)
    y_t = W_y @ h_t

    current_index = int(np.argmax(y_t))
    current_char = index_to_char[current_index]

    if current_char == end_token:
        break
    decoded_chars.append(current_char)

In [46]:
message = "".join(decoded_chars)
print("Повідомлення:")
print(message)

Повідомлення:
Назва виконавця: https://www.youtube.com/watch?v=6wCgZh-nczYw.youtube.com/watch?v=6wCgZh-nczY


In [55]:
class RNN(nn.Module):
    def __init__(self, embedding, W_h, W_h_bias, U_h, W_y):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding)
        self.W_h = nn.Parameter(W_h, requires_grad=False)
        self.W_h_bias = nn.Parameter(W_h_bias, requires_grad=False)
        self.U_h = nn.Parameter(U_h, requires_grad=False)
        self.W_y = nn.Parameter(W_y, requires_grad=False)

    def forward(self, input_index, h_previous):
        x_t = self.embedding(input_index)
        h_t = torch.relu(self.W_h @ x_t + self.W_h_bias + self.U_h @ h_previous)
        y_t = self.W_y @ h_t
        return y_t, h_t

In [56]:
model = RNN(torch.tensor(embedding), torch.tensor(W_h), torch.tensor(W_h_bias), torch.tensor(U_h), torch.tensor(W_y))
model.eval()

RNN(
  (embedding): Embedding(133, 48)
)

In [57]:
current_index = torch.tensor(char_to_index[start_token], dtype=torch.long)
h_t = torch.zeros(160, dtype=torch.float32)

decoded_chars = []

In [58]:
with torch.no_grad():
    for step in range(max_steps):
        y_t, h_t = model(current_index, h_t)

        current_index = torch.argmax(y_t)
        current_char = index_to_char[current_index.item()]

        if current_char == end_token:
          break
        decoded_chars.append(current_char)

In [59]:
message = "".join(decoded_chars)
print("Повідомлення (PyTorch):")
print(message)

Повідомлення (PyTorch):
Назва виконавця: https://www.youtube.com/watch?v=6wCgZh-nczY
